In [1]:
import pandas as pd

data = pd.read_csv('../crawling/CVF Conferences Dataset.csv')
data

,CVF Conferences,Title,Abstract
0,CVPR 2013,\nDeformable Spatial Pyramid Matching for Fast...,\nWe introduce a fast deformable spatial pyram...
1,CVPR 2013,\nMulti-target Tracking by Rank-1 Tensor Appro...,\nIn this paper we formulate multi-target trac...
2,CVPR 2013,\nWeakly-Supervised Dual Clustering for Image ...,"\nIn this paper, we propose a novel Weakly-Sup..."
3,CVPR 2013,\nReconstructing Loopy Curvilinear Structures ...,\nWe propose a novel approach to automated del...
4,CVPR 2013,\nIt's Not Polite to Point: Describing People ...,\nVisual attributes are powerful features for ...
...,...,...,...
21141,ICCV 2023,\n PVT++: A Simple End-to-End Latency-A...,\n Visual object tracking is essential to i...
21142,ICCV 2023,\n Hard No-Box Adversarial Attack on Sk...,"\n Recently, methods for skeleton-based hum..."
21143,ICCV 2023,\n Bird's-Eye-View Scene Graph for Visi...,"\n Vision-language navigation (VLN), which ..."
21144,ICCV 2023,\n Towards Attack-tolerant Federated Le...,\n Federated learning is used to train a sh...


In [2]:
from bs4 import BeautifulSoup
import re

def preprocessing(text):
    new_text = re.sub(r'[\r\n\t]', '', text)
    return BeautifulSoup(new_text, "html.parser").get_text()

In [3]:
for col in data.columns:
    data[col] = data[col].astype(str).apply(preprocessing)
data

/var/folders/_l/mh9hlz0s03xg8z3s309b4qgr0000gn/T/ipykernel_3644/2102377892.py:6: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  return BeautifulSoup(new_text, "html.parser").get_text()


,CVF Conferences,Title,Abstract
0,CVPR 2013,Deformable Spatial Pyramid Matching for Fast D...,We introduce a fast deformable spatial pyramid...
1,CVPR 2013,Multi-target Tracking by Rank-1 Tensor Approxi...,In this paper we formulate multi-target tracki...
2,CVPR 2013,Weakly-Supervised Dual Clustering for Image Se...,"In this paper, we propose a novel Weakly-Super..."
3,CVPR 2013,Reconstructing Loopy Curvilinear Structures Us...,We propose a novel approach to automated delin...
4,CVPR 2013,It's Not Polite to Point: Describing People wi...,Visual attributes are powerful features for ma...
...,...,...,...
21141,ICCV 2023,PVT++: A Simple End-to-End Latency-Awa...,Visual object tracking is essential to int...
21142,ICCV 2023,Hard No-Box Adversarial Attack on Skel...,"Recently, methods for skeleton-based human..."
21143,ICCV 2023,Bird's-Eye-View Scene Graph for Vision...,"Vision-language navigation (VLN), which en..."
21144,ICCV 2023,Towards Attack-tolerant Federated Lear...,Federated learning is used to train a shar...


In [4]:
from nltk import word_tokenize
from nltk.corpus import stopwords

stopWords = set(stopwords.words('english'))
def removeStopwords(text):
    text = [w for w in word_tokenize(text) if not w in stopWords]
    return ' '.join(text)

data['Abstract'] = data['Abstract'].apply(removeStopwords)
data

,CVF Conferences,Title,Abstract
0,CVPR 2013,Deformable Spatial Pyramid Matching for Fast D...,We introduce fast deformable spatial pyramid (...
1,CVPR 2013,Multi-target Tracking by Rank-1 Tensor Approxi...,In paper formulate multi-target tracking ( MTT...
2,CVPR 2013,Weakly-Supervised Dual Clustering for Image Se...,"In paper , propose novel Weakly-Supervised Dua..."
3,CVPR 2013,Reconstructing Loopy Curvilinear Structures Us...,We propose novel approach automated delineatio...
4,CVPR 2013,It's Not Polite to Point: Describing People wi...,Visual attributes powerful features many diffe...
...,...,...,...
21141,ICCV 2023,PVT++: A Simple End-to-End Latency-Awa...,Visual object tracking essential intelligent r...
21142,ICCV 2023,Hard No-Box Adversarial Attack on Skel...,"Recently , methods skeleton-based human activi..."
21143,ICCV 2023,Bird's-Eye-View Scene Graph for Vision...,"Vision-language navigation ( VLN ) , entails a..."
21144,ICCV 2023,Towards Attack-tolerant Federated Lear...,Federated learning used train shared model dec...


In [5]:
targets = data['CVF Conferences']

In [6]:
from bertopic import BERTopic
from bertopic.representation import TextGeneration
from transformers.pipelines import pipeline

embedding_model = pipeline("feature-extraction", model="distilbert-base-cased")
generator = pipeline('text2text-generation', model='google/flan-t5-large')
prompt = "Can you determine the subject or theme represented by these keywords: [KEYWORDS]?"
representation = TextGeneration(generator, prompt)

topic_model = BERTopic(embedding_model=embedding_model, representation_model={'Flan-T5': representation}, calculate_probabilities=True, verbose=True)
topic, probs = topic_model.fit_transform(data['Abstract'])

100%|██████████| 21146/21146 [20:16<00:00, 17.38it/s]
2023-11-27 18:08:37,180 - BERTopic - Transformed documents to Embeddings
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2023-11-27 18:08:52,241 - BERTopic - Reduced dimensionality
2023-11-27 18:08:54,139 - BERTopic - Clustered reduced embeddings
100%|██████████| 5/5 [00:03<00:00,  1.52it/s]


In [7]:
topic_model.generate_topic_labels()

['-1_the_we_in',
 '0_the_we_image',
 '1_adversarial_attacks_attack',
 '2_calibration_confidence_calibrated',
 '3_spoofing_anti_face']

In [8]:
topic_model.visualize_barchart(top_n_topics=10, n_words=10)

In [9]:
topics_per_class = topic_model.topics_per_class(data['Abstract'], classes=targets)
topic_model.visualize_topics_per_class(topics_per_class, top_n_topics=10)

21it [00:01, 18.57it/s]


In [10]:
all_topics = topic_model.get_topic_info()
all_topics

,Topic,Count,Name,Representation,Flan-T5,Representative_Docs
0,-1,1308,-1_the_we_in_based,"[the, we, in, based, model, image, network, pr...","[method, , , , , , , , , ]",[How effectively learn temporal variation targ...
1,0,19457,0_the_we_image_model,"[the, we, image, model, in, learning, method, ...","[learning based on image classification, , , ,...",[Sensitivity severe occlusion large view angle...
2,1,354,1_adversarial_attacks_attack_examples,"[adversarial, attacks, attack, examples, pertu...","[adversarial models, , , , , , , , , ]",[Adversarial attacks mislead deep neural netwo...
3,2,15,2_calibration_confidence_calibrated_predictions,"[calibration, confidence, calibrated, predicti...","[stochastic processes, , , , , , , , , ]",[Model calibration usually requires optimizing...
4,3,12,3_spoofing_anti_face_spoof,"[spoofing, anti, face, spoof, deepfake, attack...","[cyber security, , , , , , , , , ]",[Face anti-spoofing crucial prevent face recog...
